### **Notebook 6 (etapa 4): Prueba de robustez sobre fuga de pacientes**

In [ ]:
"""
Descripción:
    Script de preparación de datos para la Prueba de Robustez Final evaluando 3 variables objetivo.
    Se encarga de cargar y unificar múltiples años de datos (2019-2024), aislar exclusivamente 
    a la cohorte oncológica, limpiar identificadores de pacientes corruptos, aplicar One-Hot Encoding, 
    y realizar una partición de datos (Train/Test) agrupada por paciente. Esto último garantiza 
    que los episodios clínicos de un mismo paciente no queden divididos entre entrenamiento y prueba 
    (evitando la fuga de información o data leakage).

Entradas:
    - Archivos CSV anuales (ej. GRD_PROCESADO_2019_DERIVADAS.csv) ubicados en el directorio local.

Salidas:
    - X_train, X_test (DataFrames): Matrices de características predictoras listas para modelado.
    - (Variables en memoria preparadas para los pasos siguientes de entrenamiento y evaluación).
"""

import pandas as pd  # Permite el manejo y análisis de estructuras de datos (DataFrames)
import numpy as np  # Facilita cálculos numéricos y operaciones con vectores/matrices
import os  # Interacción con el sistema operativo (creación y verificación de rutas/directorios)
import gc  # Recolección de basura (Garbage Collector) para gestionar la memoria RAM
from sklearn.model_selection import GroupShuffleSplit  # Divisor de datos que agrupa muestras según una llave (ej. ID paciente)
import xgboost as xgb  # Algoritmo de ensamble avanzado (importado para etapas posteriores del script)
from sklearn.ensemble import RandomForestClassifier  # Algoritmo de ensamble (importado para etapas posteriores)
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score  # Métricas de evaluación de rendimiento
from sklearn.preprocessing import label_binarize  # Convierte etiquetas multiclase a formato binario (One-vs-Rest)

# 1. Configuración de rutas y columnas
# Directorio donde se ubican los archivos CSV procesados y derivados
dir_entrada = "../../Datos/Datos procesados"

# Lista exhaustiva de las columnas específicas que se extraerán de los CSV para optimizar memoria
columnas_a_cargar = [
    'CIP_ENCRIPTADO', 'MORTALIDAD', 'SEVERIDAD', 'CONSUMO_RECURSOS', 
    'CATEGORIA_CANCER', 'COMORBILIDAD_PRINCIPAL', 'ES_EXTRANJERO', 
    'ES_PUEBLO_ORIGINARIO', 'TIPO_PROCEDIMIENTO', 'ESPECIALIDAD_MEDICA', 
    'REGION', 'SERVICIOINGRESO', 'SEXO', 'TIPO_DIAGNOSTICO_ONCO', 
    'TIPO_INGRESO', 'TIPO_PREVISION', 'TIPO_PROCEDENCIA', 'CANTIDAD_TRASLADOS', 
    'CARGA_ONCOLOGICA', 'DIAS_ESTADIA', 'EDAD', 'NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS'
]

# Lista de variables categóricas que deberán ser transformadas a variables dummy (One-Hot Encoding)
vars_para_ohe = [
    'COMORBILIDAD_PRINCIPAL', 'ES_EXTRANJERO', 'ES_PUEBLO_ORIGINARIO', 
    'TIPO_PROCEDIMIENTO', 'ESPECIALIDAD_MEDICA', 'REGION', 'SERVICIOINGRESO', 
    'SEXO', 'TIPO_DIAGNOSTICO_ONCO', 'TIPO_INGRESO', 'TIPO_PREVISION', 
    'TIPO_PROCEDENCIA', 'CATEGORIA_CANCER' 
]

print("=== INICIANDO PRUEBA DE ROBUSTEZ FINAL (3 TARGETS) ===")

# Inicializar lista vacía para almacenar temporalmente los dataframes de cada año
lista_df_onco = []

# 2. Cargar todos los años y filtrar solo Oncológicos
# Iterar a través de los años disponibles en el rango de los datos (2019 a 2024)
for año in range(2019, 2025):
    archivo = f"GRD_PROCESADO_{año}_DERIVADAS.csv"
    ruta = os.path.join(dir_entrada, archivo)
    
    # Verificar si el archivo correspondiente al año existe en la ruta
    if os.path.exists(ruta):
        print(f"Cargando {año}...")
        # Cargar el archivo CSV leyendo estrictamente las columnas indicadas para ahorrar RAM
        df_temp = pd.read_csv(ruta, usecols=columnas_a_cargar, low_memory=False)
        
        # Limpiar la columna de categoría de cáncer: extraer solo el texto principal antes de los dos puntos
        df_temp['CATEGORIA_CANCER'] = df_temp['CATEGORIA_CANCER'].astype(str).str.split(':').str[0].str.replace('-', '_').str.strip()
        
        # Filtrar exclusivamente los episodios clínicos de pacientes que sí padecen cáncer
        df_onco_temp = df_temp[~df_temp['CATEGORIA_CANCER'].str.contains('SIN_CANCER', na=False)].copy()
        
        # Agregar el dataframe filtrado a la lista recolectora
        lista_df_onco.append(df_onco_temp)
        
        # Liberar la memoria del dataframe temporal completo
        del df_temp
        gc.collect()

# Unificar todos los dataframes anuales en un único conjunto de datos global
df_onco_global = pd.concat(lista_df_onco, ignore_index=True)
# Limpiar la lista temporal de la memoria
del lista_df_onco
gc.collect()

# 3. LIMPIEZA DE IDs CORRUPTOS
print("\nLimpiando IDs corruptos...")
# Eliminar filas que tengan valores nulos en el identificador del paciente
df_onco_global = df_onco_global.dropna(subset=['CIP_ENCRIPTADO'])
# Estandarizar el formato del identificador como cadena de texto sin espacios en blanco
df_onco_global['CIP_ENCRIPTADO'] = df_onco_global['CIP_ENCRIPTADO'].astype(str).str.strip()
# Definir una lista de valores anómalos o de prueba que contaminan la trazabilidad
ids_corruptos = ['SIN INFORMACIÓN', 'SIN INFORMACIÃ“N', '95162030', '95162030.0', '78492052', '78492052.0', 'nan', 'NAN']
# Filtrar el dataframe reteniendo solo los pacientes con un identificador válido
df_onco_global = df_onco_global[~df_onco_global['CIP_ENCRIPTADO'].isin(ids_corruptos)]

# 4. PREPARAR MATRIZ X Y VECTORES Y
print("Aplicando One-Hot Encoding...")
# Aplicar transformación de variables categóricas a dummies (One-Hot Encoding), omitiendo la primera categoría para evitar multicolinealidad
df_onco_ohe = pd.get_dummies(df_onco_global.drop(columns=['CIP_ENCRIPTADO']), columns=vars_para_ohe, drop_first=True)
# Normalizar los nombres de las columnas resultantes (mayúsculas, reemplazar espacios y guiones)
df_onco_ohe.columns = df_onco_ohe.columns.str.replace(' ', '_').str.replace('-', '_').str.upper()

# Matriz de características (X): Remover las variables objetivo para que el modelo no haga trampa
X = df_onco_ohe.drop(columns=['MORTALIDAD', 'SEVERIDAD', 'CONSUMO_RECURSOS'])
# Vector de agrupamiento: Extraer los IDs de los pacientes para utilizarlos como llave de partición
groups = df_onco_global['CIP_ENCRIPTADO']

print(f"Total episodios limpios: {len(X):,}")
print(f"Total pacientes únicos: {groups.nunique():,}")

# 5. SPLIT ÚNICO AGRUPADO POR PACIENTE
print("\nRealizando partición GroupShuffleSplit...")
# Instanciar el separador agrupado indicando que se dejará un 20% para pruebas
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
# Generar los índices de entrenamiento y prueba respetando que un paciente completo esté en uno u otro set, nunca en ambos
train_idx, test_idx = next(gss.split(X, groups=groups))

# Crear las matrices definitivas de entrenamiento y prueba utilizando los índices generados
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

# Destruir el dataset global original y limpiar la memoria RAM para las siguientes operaciones
del df_onco_global
gc.collect()

=== INICIANDO PRUEBA DE ROBUSTEZ FINAL (3 TARGETS) ===
Cargando 2019...
Cargando 2020...
Cargando 2021...
Cargando 2022...
Cargando 2023...
Cargando 2024...

Limpiando IDs corruptos...
Aplicando One-Hot Encoding...
Total episodios limpios: 493,154
Total pacientes únicos: 284,547

Realizando partición GroupShuffleSplit...


0

In [ ]:
# Función auxiliar para calcular métricas multiclase y binarias de forma elegante
def evaluar_metricas(y_true, y_pred, y_prob, es_multiclase=False):
    """
    Descripción:
        Calcula de manera centralizada y dinámica las métricas de rendimiento predictivo 
        (F1-Macro, F1-Clase 1, AUC-ROC y AUPRC). Adapta el cálculo matemáticamente 
        dependiendo de si se trata de un problema de clasificación binaria o multiclase 
        (utilizando estrategias One-vs-Rest ponderadas para multiclase).

    Entradas:
        - y_true (Series/array): Etiquetas reales (ground truth) del conjunto de prueba.
        - y_pred (Series/array): Predicciones duras (clases discretas finales) emitidas por el modelo.
        - y_prob (Series/array): Probabilidades continuas emitidas por el modelo.
        - es_multiclase (bool): Bandera (flag) que indica si el problema tiene más de dos clases.

    Salidas:
        - Tupla de métricas (float, float/None, float, float): 
          Retorna estructuradamente (F1-Macro, F1-Clase 1 [Solo si es binario], AUC-ROC, AUPRC).
    """
    # Calcular siempre el F1-Score promediado de forma Macro (trata todas las clases con la misma importancia)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    
    # --- FLUJO DE MÉTRICAS BINARIAS ---
    if not es_multiclase:
        # Calcular F1-Score exclusivamente para la clase positiva/minoritaria
        f1_clase1 = f1_score(y_true, y_pred, pos_label=1) # <-- AQUÍ AGREGAMOS LA CLASE 1
        # Calcular el Área Bajo la Curva ROC clásica
        auc = roc_auc_score(y_true, y_prob)
        # Calcular el Área Bajo la Curva Precision-Recall (métrica ideal para clases desbalanceadas)
        auprc = average_precision_score(y_true, y_prob)
        
        # Retornar las 4 métricas calculadas
        return f1_macro, f1_clase1, auc, auprc
        
    # --- FLUJO DE MÉTRICAS MULTICLASE ---
    else:
        # Calcular AUC multiclase utilizando la estrategia One-vs-Rest (OvR) ponderada por soporte
        auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted')
        
        # Identificar las clases únicas presentes en la variable objetivo
        clases = np.unique(y_true)
        # Binarizar el y_true en múltiples columnas (requisito matemático para calcular AUPRC multiclase)
        y_true_bin = label_binarize(y_true, classes=clases)
        
        # Calcular AUPRC multiclase ponderado por la cantidad real de muestras de cada clase
        auprc = average_precision_score(y_true_bin, y_prob, average='weighted')
        
        # Retornar métricas (F1_clase1 es None porque no aplica lógicamente en este contexto multiclase)
        return f1_macro, None, auc, auprc


# Imprimir separadores y encabezado visual en consola para organizar la salida
print("\n" + "="*50)
print("ENTRENAMIENTO Y EVALUACIÓN POR TARGET")
print("="*50)

# ==========================================
# TARGET 1: MORTALIDAD (BINARIO) -> Random Forest
# ==========================================
print("\n1. Entrenando MORTALIDAD (Random Forest)...")
# Aislar la variable objetivo de mortalidad del dataset preparado (OHE)
y_mort = df_onco_ohe['MORTALIDAD']
# Dividir el vector objetivo utilizando los índices agrupados por paciente (sin fuga de datos)
y_train_m, y_test_m = y_mort.iloc[train_idx], y_mort.iloc[test_idx]

# Instanciar el modelo de Bosques Aleatorios con los hiperparámetros estables encontrados previamente
modelo_rf = RandomForestClassifier(
    n_estimators=500, max_depth=35, min_samples_split=10, 
    class_weight='balanced', n_jobs=-1, random_state=42
)
# Ajustar el modelo a los datos de entrenamiento
modelo_rf.fit(X_train, y_train_m)

# Predecir las clases duras sobre la matriz X_test oculta
y_pred_m = modelo_rf.predict(X_test)
# Predecir las probabilidades, extrayendo únicamente la columna de la clase positiva (índice 1)
y_prob_m = modelo_rf.predict_proba(X_test)[:, 1]

# Ejecutar la evaluación de métricas utilizando el flujo binario de nuestra función auxiliar
f1_m_macro, f1_m_c1, auc_m, auprc_m = evaluar_metricas(y_test_m, y_pred_m, y_prob_m, es_multiclase=False)
# Imprimir el rendimiento final alcanzado en la predicción de Mortalidad
print(f"-> F1-Macro: {f1_m_macro:.4f} | F1-Clase 1: {f1_m_c1:.4f} | AUC-ROC: {auc_m:.4f} | AUPRC: {auprc_m:.4f}")


# ==========================================
# TARGET 2: SEVERIDAD (MULTICLASE) -> XGBoost
# ==========================================
print("\n2. Entrenando SEVERIDAD (XGBoost Multiclase)...")
# Aislar la variable objetivo de severidad
y_sev = df_onco_ohe['SEVERIDAD']
# Dividir el vector objetivo respetando la agrupación por pacientes
y_train_s, y_test_s = y_sev.iloc[train_idx], y_sev.iloc[test_idx]

# Instanciar el modelo XGBoost con los hiperparámetros óptimos para esta variable
modelo_xgb_sev = xgb.XGBClassifier(
    learning_rate=0.3, max_depth=10, tree_method='hist', 
    n_jobs=-1, random_state=42
)
# Entrenar el modelo con las características de los episodios de entrenamiento
modelo_xgb_sev.fit(X_train, y_train_s)

# Generar predicciones (clase) y todas las probabilidades de clases (matriz completa)
y_pred_s = modelo_xgb_sev.predict(X_test)
y_prob_s = modelo_xgb_sev.predict_proba(X_test)

# Evaluar el desempeño utilizando el flujo multiclase (es_multiclase=True)
f1_s_macro, _, auc_s, auprc_s = evaluar_metricas(y_test_s, y_pred_s, y_prob_s, es_multiclase=True)
# Imprimir los resultados consolidados para Severidad
print(f"-> F1-Macro: {f1_s_macro:.4f} | AUC-ROC: {auc_s:.4f} | AUPRC: {auprc_s:.4f}")


# ==========================================
# TARGET 3: CONSUMO DE RECURSOS (MULTICLASE) -> XGBoost
# ==========================================
print("\n3. Entrenando CONSUMO DE RECURSOS (XGBoost Multiclase)...")
# Aislar la variable objetivo de consumo de recursos
y_cons = df_onco_ohe['CONSUMO_RECURSOS']
# Aplicar la separación por índices agrupados
y_train_c, y_test_c = y_cons.iloc[train_idx], y_cons.iloc[test_idx]

# Configurar XGBoost con los hiperparámetros estables
modelo_xgb_cons = xgb.XGBClassifier(
    learning_rate=0.3, max_depth=10, tree_method='hist', 
    n_jobs=-1, random_state=42
)
# Realizar el ajuste matemático del modelo
modelo_xgb_cons.fit(X_train, y_train_c)

# Inferir resultados (clase y probabilidades) sobre el set de prueba
y_pred_c = modelo_xgb_cons.predict(X_test)
y_prob_c = modelo_xgb_cons.predict_proba(X_test)

# Calcular indicadores globales para el problema multiclase
f1_c_macro, _, auc_c, auprc_c = evaluar_metricas(y_test_c, y_pred_c, y_prob_c, es_multiclase=True)
# Reportar resultados finales para Consumo de Recursos
print(f"-> F1-Macro: {f1_c_macro:.4f} | AUC-ROC: {auc_c:.4f} | AUPRC: {auprc_c:.4f}")

# Imprimir mensaje de cierre del script
print("\n=== FIN DE LA PRUEBA DE ROBUSTEZ ===")


ENTRENAMIENTO Y EVALUACIÓN POR TARGET

1. Entrenando MORTALIDAD (Random Forest)...
-> F1-Macro: 0.6896 | F1-Clase 1: 0.4394 | AUC-ROC: 0.9231 | AUPRC: 0.4342

2. Entrenando SEVERIDAD (XGBoost Multiclase)...
-> F1-Macro: 0.7712 | AUC-ROC: 0.9066 | AUPRC: 0.8145

3. Entrenando CONSUMO DE RECURSOS (XGBoost Multiclase)...
-> F1-Macro: 0.7431 | AUC-ROC: 0.8950 | AUPRC: 0.8635

=== FIN DE LA PRUEBA DE ROBUSTEZ ===
